# MeteoScreening from database (influxdb)

---
**Notebook version**: `12` (25 Sep 2026)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

## ℹ️ About this notebook
Downloads raw meteo data from InfluxDB, screens and corrects it at high resolution, resamples it to `RESAMPLING_FREQ` and uploads the result back to the database. Screening uses [`StepwiseMeteoScreeningDb`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb). Download and upload use `InfluxIO`, which needs `uv sync --group db`.

**Screening is stepwise.** Run a test, check its preview, then commit it with `mscr.addflag()`. At the end all committed flags are combined into one quality flag `QCF`.

**Run All** does not change data with the example settings: corrections and tests with data-specific settings are commented out. It does upload the result, see *Upload data to database*.

## ⏱️ Timestamp convention (important)
The database stores every timestamp in **UTC** as **`TIMESTAMP_END`**, the end of the averaging interval. Set `TIMEZONE_OFFSET_TO_UTC_HOURS` to the timezone the raw data was logged in (for example `1` for UTC+01:00). The day/night split during screening and the timestamps written back to the database depend on it. The offset is applied the same way on download and on upload:

| Stage | Timezone | Convention | Done by |
|---|---|---|---|
| Database | UTC | `TIMESTAMP_END` | InfluxDB |
| After `dbc.download(..., timezone_offset_to_utc_hours=N)` | local (UTC+N) | `TIMESTAMP_END` | InfluxIO |
| During screening | local | `TIMESTAMP_MIDDLE` (converted internally) | `StepwiseMeteoScreeningDb` |
| After `mscr.resample()` | local | back to `TIMESTAMP_END` | diive |
| After `dbc.upload_singlevar(..., timezone_offset_to_utc_hours=N)` | UTC | `TIMESTAMP_END` | InfluxIO |

Manual removal takes timestamps as they appear in the downloaded data: local time, `TIMESTAMP_END`.

## ✏️ User settings (please adjust)

Adjust these before running. What each setting means:

**Site**
- `SITE`, `SITE_LAT`, `SITE_LON`: site ID and coordinates. The coordinates set the day/night split used during screening.

**Variables to screen**
- `FIELDS`: variable name(s) exactly as stored in the database (the InfluxDB `_field`). Several are allowed.
- `MEASUREMENT`: exactly **one** measurement that groups those variables.

**Time range to screen**
- `START`: first timestamp to screen. It **is** included.
- `STOP`: upper bound. It is **not** included.

**Data settings**
- `TIMEZONE_OFFSET_TO_UTC_HOURS`: the timestamp knob, see *Timestamp convention*. It must match how the raw data was logged.
- `DATA_VERSION`: the source data version in the database (`raw`).
- `DIRCONF`: local folder holding the database connection config.

**Resampling**
- `RESAMPLING_FREQ`: the screened high-res data is resampled to this frequency.
- `RESAMPLING_AGG`: `'mean'` or `'sum'`. Use `'sum'` only for variables that accumulate over the interval.

In [ ]:
# --- Site ---
SITE = 'ch-hon'
SITE_LAT = 47.41887
SITE_LON = 8.491318

# --- Variables to screen ---
FIELDS = [
    'TA_T1_4_2',
]
MEASUREMENT = 'TA'

# --- Time range to screen ---
START = '2026-03-01 00:00:01'  # included
STOP = '2026-04-01 00:00:01'  # not included

# --- Data settings ---
DATA_VERSION = 'raw'
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # UTC+01:00 (CET, winter time)
DIRCONF = r'path/to/configs'  # <-- must be set: folder with the database connection config
# DIRCONF = r'F:\dev\poet\configs'
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

# --- Resampling ---
RESAMPLING_FREQ = '30min'
RESAMPLING_AGG = 'mean'  # 'mean' or 'sum'

## 🤖 Auto settings

### Buckets (do not adjust)

In [ ]:
BUCKET_RAW = f'{SITE}_raw'  # source bucket
BUCKET_PROCESSED = f'{SITE}_processed'  # destination bucket
print(f'Source bucket (raw data):       {BUCKET_RAW}')
print(f'Destination bucket (processed): {BUCKET_PROCESSED}')

### Imports

In [ ]:
import warnings
from datetime import datetime

import pandas as pd
from pandas.tseries.frequencies import to_offset

import diive as dv
from diive.core.io.db.influx import InfluxIO  # needs: uv sync --group db

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
print(f"Last run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f'diive v{dv.__version__}')

## ⬇️ Download data from database

### Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

Optional. Lists all fields in the measurement, without checking the selected time range:

In [ ]:
# display(dbc.show_fields_in_measurement(bucket=BUCKET_RAW, measurement=MEASUREMENT))

### Download
Returns three objects:
- `data_simple`: high-res time series, one column per variable.
- `data_detailed`: dict `{varname: DataFrame}` with each variable's time series and its database tags. This is what the screening reads.
- `assigned_measurements`: the measurement detected per variable, as a check.

In [ ]:
%%time
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET_RAW,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

### Inspect downloaded data

In [ ]:
display(data_simple)
display(assigned_measurements)

Drop any requested variable that has no data in this period:

In [ ]:
missing = [v for v in FIELDS if v not in data_detailed]
if missing:
    print(f'No data in this period, removed from FIELDS: {missing}')
FIELDS = [v for v in FIELDS if v in data_detailed]
print(f'Data available for: {FIELDS}')

### Verify download timestamps
The timestamps should be in local time (UTC+`TIMEZONE_OFFSET_TO_UTC_HOURS`) and mark the end of each averaging interval. Compare the first and last stamps against the `START` and `STOP` you asked for.

In [ ]:
for v, frame in data_detailed.items():
    idx = frame.index
    print(f'{v}: {idx[0]} to {idx[-1]}  (index {idx.name})')
print(f'\nUTC offset applied: +{TIMEZONE_OFFSET_TO_UTC_HOURS}h, so the stamps above are local time.')

Optional. Save the full-resolution raw data to a file:

In [ ]:
# data_detailed[FIELDS[0]].to_csv('rawdata_highres.csv')

### Plot downloaded high-res data

In [ ]:
for varname, frame in data_detailed.items():
    dv.plotting.TimeSeries(series=frame[varname]).plot()

## ▶️ Start MeteoScreening with `diive`
`showplot_orig()` shows the raw series. If it needs no outlier removal, skip *Outlier detection* and *Overall quality flag QCF* and continue at *Corrections*.

In [ ]:
mscr = dv.qaqc.StepwiseMeteoScreeningDb(
    site=SITE,
    data_detailed=data_detailed,
    fields=FIELDS,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
)
mscr.showplot_orig()

## 🔍 Outlier detection
Each method has a test cell and an `mscr.addflag()` cell. Rerun the test with other settings as often as needed; `mscr.addflag()` commits the most recent one. Run only the methods your variable needs.

With `separate_day_night=True` the statistic is computed separately for daytime and nighttime, split by the site coordinates. Some methods also take separate daytime and nighttime thresholds, see the linked docs. Window sizes are time spans such as `'7D'`, converted to records at the data's time resolution, so the examples work for 1-minute and 10-minute data alike. A record count also works.

Tests with data-specific example settings are commented out, so *Run All* does not apply them. To use one, edit the settings and uncomment the test and its `mscr.addflag()` cell.

In [ ]:
mscr.start_outlier_detection()

Plot the current cleaned data at any point during detection. `interactive=True` opens a Bokeh plot in the browser instead, with hover and zoom. `showplot_orig()` and `showplot_cleaned()` take the same argument.

In [ ]:
mscr.showplot_outlier_detection_cleaned()

### Manual removal
Flags single timestamps or `[start, end]` ranges (both ends included), for example known sensor failures. Give timestamps as they appear in the downloaded data: local time, end of the averaging period.
Docs: [`flag_manualremoval_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_manualremoval_test), [`ManualRemoval`](https://diive.readthedocs.io/en/latest/api/outliers.html#diive.outliers.ManualRemoval).

In [ ]:
REMOVE_DATES = [
    # '2026-03-10 14:31:00',  # single record
    # ['2026-03-12 08:00:00', '2026-03-12 11:30:00'],  # range, both ends included
]
mscr.flag_manualremoval_test(remove_dates=REMOVE_DATES, showplot=True, verbose=True)

In [ ]:
mscr.addflag()

### Hampel filter
Flags spikes that lie far from the median of a sliding window, measured in median absolute deviations. A robust general-purpose filter.
Docs: [`flag_outliers_hampel_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_outliers_hampel_test), [`Hampel`](https://diive.readthedocs.io/en/latest/api/outliers.html#diive.outliers.Hampel).

In [ ]:
mscr.flag_outliers_hampel_test(
    window_length='7D',  # time span; one week at any data resolution
    n_sigma_daytime=5.5, n_sigma_nighttime=5.5,
    use_differencing=True, separate_day_night=True,
    repeat=True, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Z-score
Flags values more than `thres_zscore` standard deviations from the mean of the whole period.
Docs: [`flag_outliers_zscore_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_outliers_zscore_test), [`zScore`](https://diive.readthedocs.io/en/latest/api/outliers.html#diive.outliers.zScore).

In [ ]:
mscr.flag_outliers_zscore_test(
    thres_zscore=4.5, separate_day_night=True,
    repeat=True, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Z-score (rolling window)
The z-score in a moving window of `winsize` records, so it follows slow changes and flags local spikes.
Docs: [`flag_outliers_zscore_rolling_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_outliers_zscore_rolling_test), [`zScoreRolling`](https://diive.readthedocs.io/en/latest/api/outliers.html#diive.outliers.zScoreRolling).

In [ ]:
mscr.flag_outliers_zscore_rolling_test(
    thres_zscore=4.5, winsize='7D',  # time span; one week at any data resolution
    repeat=True, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Local standard deviation
Flags values more than `n_sd` standard deviations from the rolling median of `winsize` records.
Docs: [`flag_outliers_localsd_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_outliers_localsd_test), [`LocalSD`](https://diive.readthedocs.io/en/latest/api/outliers.html#diive.outliers.LocalSD).

In [ ]:
mscr.flag_outliers_localsd_test(
    separate_day_night=True, n_sd=5.5, winsize='7D',  # time span; one week at any data resolution
    constant_sd=False, repeat=False, showplot=True, verbose=True,
)

In [ ]:
mscr.addflag()

### Increments z-score
Flags records with unusually large jumps to their neighbours, based on the z-score of the differences between records. Records are not compared across gaps.
Docs: [`flag_outliers_increments_zcore_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_outliers_increments_zcore_test), [`zScoreIncrements`](https://diive.readthedocs.io/en/latest/api/outliers.html#diive.outliers.zScoreIncrements).

In [ ]:
mscr.flag_outliers_increments_zcore_test(thres_zscore=40, repeat=True, showplot=True, verbose=True)

In [ ]:
mscr.addflag()

### Local outlier factor
Flags points that sit apart from their nearest neighbours. It always flags the `contamination` fraction of records, and it is slow on high-resolution data.
Docs: [`flag_outliers_lof_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_outliers_lof_test), [`LocalOutlierFactor`](https://diive.readthedocs.io/en/latest/api/outliers.html#diive.outliers.LocalOutlierFactor).

In [ ]:
# mscr.flag_outliers_lof_test(
#     n_neighbors=30, contamination=0.01, separate_day_night=False,
#     repeat=False, n_jobs=-1, showplot=True, verbose=True,
# )

In [ ]:
# mscr.addflag()

### Absolute limits
Flags values outside `[minval, maxval]`, optionally with separate daytime and nighttime limits. The example limits fit air temperature in °C only.
Docs: [`flag_outliers_abslim_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_outliers_abslim_test), [`AbsoluteLimits`](https://diive.readthedocs.io/en/latest/api/outliers.html#diive.outliers.AbsoluteLimits).

In [ ]:
# mscr.flag_outliers_abslim_test(minval=-18, maxval=50, showplot=True, verbose=True)

In [ ]:
# mscr.addflag()

### Trim low
Flags values below `lower_limit` and the same number of the highest values, in the chosen period. Check the preview: a high `lower_limit` can remove a large share of the data.
Docs: [`flag_outliers_trim_low_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_outliers_trim_low_test), [`TrimLow`](https://diive.readthedocs.io/en/latest/api/outliers.html#diive.outliers.TrimLow).

In [ ]:
# mscr.flag_outliers_trim_low_test(
#     trim_daytime=False, trim_nighttime=True, lower_limit=10,
#     showplot=True, verbose=True,
# )

In [ ]:
# mscr.addflag()

### Missing values
Not an outlier test. It flags missing records so they are counted in `QCF`, and needs no `addflag()`.
Docs: [`flag_missingvals_test`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.flag_missingvals_test).

In [ ]:
mscr.flag_missingvals_test(verbose=True)

### Overall quality flag QCF
Combines all committed flags into `QCF` (0 = good, 1 = marginal, 2 = bad) and filters the series. Run it before corrections and resampling.
Docs: [`finalize_outlier_detection`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.finalize_outlier_detection).

In [ ]:
mscr.finalize_outlier_detection()

#### Reports

In [ ]:
mscr.report_outlier_detection_qcf_evolution()

In [ ]:
mscr.report_outlier_detection_qcf_flags()

In [ ]:
mscr.report_outlier_detection_qcf_series()

#### Plots

In [ ]:
mscr.showplot_outlier_detection_qcf_heatmaps()
# mscr.showplot_outlier_detection_qcf_timeseries()

## 🔧 Corrections
Applied to the QCF-filtered high-res data. Run only what applies to your variable.

All calls are commented out, because they change the data with example settings and *Run All* would apply them. Edit the settings, then uncomment the call.

In [ ]:
mscr.showplot_cleaned()

### Remove radiation zero offset
For shortwave radiation and PAR. Sets nighttime values to zero and removes the daily nighttime offset from daytime values.
Docs: [`correction_remove_nighttime_zero_offset`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.correction_remove_nighttime_zero_offset).

In [ ]:
# mscr.correction_remove_nighttime_zero_offset()

### Remove relative humidity offset
Removes the offset so relative humidity does not exceed 100%.
Docs: [`correction_remove_relativehumidity_offset`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.correction_remove_relativehumidity_offset).

In [ ]:
# mscr.correction_remove_relativehumidity_offset()

### Set to maximum or minimum threshold
Sets values above or below a threshold to the threshold. The result looks plausible but was not measured, so removing the value is usually better.
Docs: [`correction_setto_max_threshold`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.correction_setto_max_threshold), [`correction_setto_min_threshold`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.correction_setto_min_threshold).

In [ ]:
# mscr.correction_setto_max_threshold(threshold=30)
# mscr.correction_setto_min_threshold(threshold=-5)

### Set a time range to a value
Sets all records in the given date ranges to one value.
Docs: [`correction_setto_value`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.correction_setto_value).

In [ ]:
DATES = [
    ['2022-04-01', '2022-04-05'],
]
# mscr.correction_setto_value(dates=DATES, value=3.7, verbose=1)
# mscr.showplot_cleaned(interactive=False)

### Set exact values to missing
Sets records equal to the given values to missing, for example a stuck reading. The first cell lists the most frequent values and only reads the data.
Docs: [`correction_set_exact_value_to_missing`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.correction_set_exact_value_to_missing).

In [ ]:
for ff in mscr.fields:
    vc = mscr.series_hires_cleaned[ff].value_counts()
    print(f'--- {ff} (top 20 of {mscr.series_hires_cleaned[ff].count()} records) ---')
    print(vc.head(20))

In [ ]:
# mscr.correction_set_exact_value_to_missing(values=[0])
# mscr.showplot_cleaned(interactive=False)

## 📈 Analyses (optional)

### Check for timestamp shifts
For radiation variables. Correlates the measured series with potential radiation day by day; a steady offset points to a shifted timestamp.
Docs: [`analysis_potential_radiation_correlation`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.analysis_potential_radiation_correlation).

In [ ]:
# _ = mscr.analysis_potential_radiation_correlation(
#     utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS, mincorr=0.7, showplot=True)

## 🔁 Resampling

### Resample
Resamples the screened high-res series to `RESAMPLING_FREQ`. The output uses `TIMESTAMP_END` again, ready for upload.
Docs: [`resample`](https://diive.readthedocs.io/en/latest/api/qaqc.html#diive.qaqc.StepwiseMeteoScreeningDb.resample).

In [ ]:
mscr.resample(to_freqstr=RESAMPLING_FREQ, agg=RESAMPLING_AGG, mincounts_perc=.25)
mscr.showplot_resampled()

### Check the resampled time resolution

In [ ]:
for v in mscr.resampled_detailed.keys():
    freq = dv.times.DetectFrequency(index=mscr.resampled_detailed[v].index, verbose=True).get()
    status = 'PASSED' if to_offset(freq) == to_offset(RESAMPLING_FREQ) else '(!) FAILED'
    print(f'{status} - {v}: {freq}')

## ⬆️ Upload data to database

**Existing screened data are overwritten.** Uploads to the processed bucket as data version `meteoscreening_diive`. With `delete_from_db_before_upload=True` the upload first deletes the existing values of the uploaded variable (same site, measurement, name and data version) between its first and last uploaded timestamp, then writes the new ones. Raw data, other variables and other data versions are not touched.

In [ ]:
print(f'Uploading to bucket {BUCKET_PROCESSED}')
for v in mscr.resampled_detailed.keys():
    dbc.upload_singlevar(
        to_bucket=BUCKET_PROCESSED,
        to_measurement=assigned_measurements[v],
        var_df=mscr.resampled_detailed[v],
        timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
        delete_from_db_before_upload=True,
    )

### Verify upload
Download the uploaded data again and check its time resolution and timestamps. They should be local `TIMESTAMP_END`, matching what was screened.

In [ ]:
# Fresh names so the screened originals are not overwritten:
check_simple, check_detailed, check_measurements = dbc.download(
    bucket=BUCKET_PROCESSED,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version='meteoscreening_diive',
)
check_simple

In [ ]:
for v in check_detailed.keys():
    idx = check_detailed[v].index
    freq = dv.times.DetectFrequency(index=idx, verbose=True).get()
    status = 'PASSED' if to_offset(freq) == to_offset(RESAMPLING_FREQ) else '(!) FAILED'
    print(f'{status} - {v}: freq={freq}, first={idx[0]}, last={idx[-1]}')

## ✅ End of notebook

In [ ]:
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")